In [ ]:
import sys
from pathlib import Path
project_root = Path(__file__).resolve().parents[0] if '__file__' in globals() else Path().resolve().parents[0]
sys.path.insert(0, str(project_root))

### Make Encodings

In [ ]:
from tomato.utils import load_critic_review_df, tomato_data_path
df_full = load_critic_review_df()
df_unif = df_full\
    [~df_full.review_content.isna()]\
    .sample(1000,random_state=42)

In [ ]:
from tomato.encoding import bert_encode_reviews

texts = df_unif.review_content.astype(str).tolist()
ids = df_unif.index.tolist()
df_enc = bert_encode_reviews(texts, ids, "bert-base-uncased")

In [ ]:
df_enc.to_parquet(tomato_data_path() / 'encoding_unif5k.parquet', compression='snappy')

### Test Pooling

In [ ]:
from tomato.utils import tomato_data_path
import pandas as pd

df_enc = pd.read_parquet(tomato_data_path() / 'encoding_unif5k.parquet')

In [ ]:
dim_cols = [c for c in df_enc.columns if c.startswith('dim_')]

In [ ]:
cls_vectors = df_enc[df_enc['token_id'] == 0][dim_cols].values

In [ ]:
import numpy as np
mean_vectors = np.vstack(
    df_enc\
        .groupby('review_id')\
        .apply(lambda g: np.average(
            g[dim_cols].values, weights=g.attention_mask, axis=0))\
        .values)

In [ ]:
from tomato.encoding import subtract_pcs

X0 = subtract_pcs(cls_vectors, 0, -1)
X = X0[0:100,:]


In [ ]:
from ripser import ripser
from persim import plot_diagrams

X = mean_vectors

results = ripser(X,2,distance_matrix=False)
diagrams = results['dgms']
plot_diagrams(diagrams, show=True)

### Test Metrics

In [ ]:
from tomato.metrics import pdist2
D = pdist2(X,X,"cosine")

In [ ]:
x = df_enc.groupby('review_id')
x.groups

In [ ]:
dim_cols = [c for c in df_enc.columns if c.startswith('dim_')]
group_dstrbs = [group[dim_cols].values for _, group in df_enc.groupby('review_id')]

In [ ]:
from tomato.metrics import wasserstein_distance
wasserstein_distance(group_dstrbs[0],group_dstrbs[1])

In [ ]:

pdist2(group_dstrbs[0],group_dstrbs[1]).shape

In [ ]:
import ot

X,Y = group_dstrbs[0],group_dstrbs[1]
n, m = X.shape[0], Y.shape[0]
# Cost matrix
C = pdist2(X, Y)
a = np.ones(n) / n
b = np.ones(m) / m
ot.emd2(a, b, C)

In [ ]:
len(group_dstrbs)

In [ ]:
from tomato.metrics import wasserstein_distances

df_enc_small = df_enc[df_enc.review_id.isin(df_enc.review_id.drop_duplicates().head(100))]
D = wasserstein_distances(df_enc_small)

In [ ]:
from tomato.metrics import wasserstein_distances_parallel
D = wasserstein_distances_parallel(df_enc)

In [ ]:
from tomato.metrics import wasserstein_distances_parallel
D = wasserstein_distances_parallel(df_enc_small)

### Wasserstein is hard :(

In [ ]:
import time, math, gc
from typing import Tuple, List, Dict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from numpy.linalg import eigh
from tqdm.auto import tqdm

# utils
def _mean_var_np(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """mean & diagonal variance (unbiased)."""
    return X.mean(0), X.var(0, ddof=1)

def _mean_cov_np(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mu = X.mean(0)
    Xc = X - mu
    cov = (Xc.T @ Xc) / max(len(X) - 1, 1)
    return mu, cov

def _sqrtm_eigh(Σ: np.ndarray) -> np.ndarray:
    lam, V = eigh(Σ)
    return (V * np.sqrt(np.clip(lam, 0, None))) @ V.T

# algs
def diag_gpu(df, device="cuda:0") -> np.ndarray:
    """
    2‑Wasserstein with diagonal Σ, fully‑vectorised.
    Runs on CPU if CUDA not available.
    """
    dev = device if torch.cuda.is_available() else "cpu"
    dim_cols = [c for c in df.columns if c.startswith("dim_")]

    mus, sigs = [], []
    for _, g in tqdm(df.groupby("review_id"), desc="diag_gpu||stats"):
        mu, var = _mean_var_np(g[dim_cols].values.astype("float32"))
        mus.append(mu)
        sigs.append(np.sqrt(var))

    mu_t = torch.as_tensor(np.stack(mus), device=dev)
    sig_t = torch.as_tensor(np.stack(sigs), device=dev)

    d_mu = torch.cdist(mu_t, mu_t)          # ||Δμ||
    d_sig = torch.cdist(sig_t, sig_t)       # ||Δ√σ||
    D     = torch.sqrt(d_mu**2 + d_sig**2)

    return D.cpu().numpy()

def diag_cpu(df) -> np.ndarray:
    """same as diag_gpu but forced to CPU."""
    return diag_gpu(df, device="cpu")

def full_cpu(df) -> np.ndarray:
    """
    Full‑covariance closed‑form (NumPy CPU) with outer loop progress.
    O(n² d³) – use on ≤300 reviews or reduced d.
    """
    dim_cols = [c for c in df.columns if c.startswith("dim_")]
    stats = [
        _mean_cov_np(g[dim_cols].values.astype("float32"))
        for _, g in tqdm(df.groupby("review_id"), desc="full_cpu||stats")
    ]

    n = len(stats)
    mu_list  = [m for m, _ in stats]
    cov_list = [c for _, c in stats]
    sqrt_cov_list = [
        _sqrtm_eigh(cov)
        for cov in tqdm(cov_list, desc="full_cpu||sqrt")
    ]

    D = np.zeros((n, n), dtype=np.float32)
    for i in tqdm(range(n), desc="full_cpu||pairs"):
        mu_i, cov_i, sqrt_cov_i = mu_list[i], cov_list[i], sqrt_cov_list[i]
        for j in range(i + 1, n):
            mu_j, cov_j = mu_list[j], cov_list[j]
            delta_mu2 = np.dot(mu_i - mu_j, mu_i - mu_j)
            M = sqrt_cov_i @ cov_j @ sqrt_cov_i
            lam = eigh(M, eigvals_only=True)
            cross = 2.0 * np.sqrt(np.clip(lam, 0, None)).sum()
            w2sq = delta_mu2 + np.trace(cov_i) + np.trace(cov_j) - cross
            D[i, j] = D[j, i] = math.sqrt(max(w2sq, 0.0))
    return D

algos: Dict[str, callable] = {
    "diag_gpu": diag_gpu,   # fastest on your RTX 3080 Ti
    "diag_cpu": diag_cpu,
    "full_cpu": full_cpu,
}

# bench
def time_algos(df, repeats: int = 1) -> pd.DataFrame:
    rows: List[Dict] = []
    for name, fn in algos.items():
        print(f"▶ {name}")
        gc.collect()
        torch.cuda.empty_cache()
        t0 = time.perf_counter()
        for _ in tqdm(range(repeats), desc=f"{name}||runs"):
            D = fn(df)
        dt = (time.perf_counter() - t0) / repeats
        rows.append(dict(algorithm=name, seconds=dt, shape=D.shape))
    return pd.DataFrame(rows).sort_values("seconds")


bench = time_algos(df_enc, repeats=1)
display(bench.style.format({"seconds": "{:.2f}"}))
